In [1]:
import sys
print(sys.executable)
print(sys.version)

c:\Users\Administrateur\Documents\M2i\.venv\Scripts\python.exe
3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]


In [2]:
import pandas as pd
import numpy as np
import json

In [3]:
# global def
import os

food_parquet = "../../data/food.parquet"
food_light_parquet = "../../data/food_light_fr.parquet"
food_light_extended_parquet = "../../data/food_light_extended_fr.parquet"

nutri_score_list = ['a', 'b', 'c', 'd', 'e']

def filter_by_country(value, country):
    try:
        # Transforme la chaîne JSON en véritable liste Python
        list_array = list(value)
        return country in list_array
    except (json.JSONDecodeError, TypeError):
        # Gestion des erreurs si le JSON est malformé ou s'il y a un NaN
        return False

def filter_by_nutriscore(score):
        return score in nutri_score_list


In [ ]:
if os.path.exists(food_light_parquet):
    off_light_df_fr = pd.read_parquet(food_light_parquet)
# elif os.path.exists(food_light_extended_parquet):
#     off_light_extended_df_fr = pd.read_parquet(food_light_extended_parquet)
else:
    off_nutriments_tags_df = pd.read_parquet(food_parquet, columns=["code", "nutriments", "countries_tags"])
    off_light_df_fr = off_nutriments_tags_df[off_nutriments_tags_df["countries_tags"].apply(lambda x: filter_by_country(x, "en:france"))]
    off_light_df_fr.to_parquet(food_light_parquet, index=False)

print(len(off_light_df_fr))

In [ ]:
counts = off_light_df_fr.groupby("code")["code"].count()
duplicate_codes = counts[counts > 1].index
# off_light_df_fr[off_light_df_fr["code"].isin(duplicate_codes)]


In [ ]:
print(f"Nombre de code en doublon: {len(duplicate_codes)}")

In [ ]:
unique_name = set()

def affiche_depuis_liste(liste_nutriments):
    if not isinstance(liste_nutriments, (np.ndarray, list)):
        return type(liste_nutriments)
    
    for nutriment in liste_nutriments:
        if isinstance(nutriment, dict):
            unique_name.add(nutriment.get('name'))

off_light_df_fr['nutriments'].apply(lambda x: affiche_depuis_liste(x))


In [ ]:
for name in unique_name:
    print(name)

In [ ]:
my_set = ("vitamin-a", "vitamin-b1", "vitamin-b2", "vitamin-b6", "vitamin-b9", "vitamin-b12", "vitamin-c", "vitamin-d", "vitamin-e", "vitamin-k", "vitamin-pp", "vitamine-h", "energy", "energy-kj", "energy-kcal", "fat", "saturated-fat", "carbohydrates", "sugars", "fiber", "proteins", "salt", "sodium", "cholesterol", "calcium", "iron", "magnesium", "potassium", "zinc", "omega-3-fat", "omega-6-fat", "omega-9-fat", "caffeine", "alcohol", "water")
for name in my_set:
    print(name)

In [ ]:
nutriments_categorized = dict()
nutriments_categorized["nutriments"] = []
nutriments_categorized["nutriments"].append({"Vitamines": [
	"vitamin-a", 
    "vitamin-b1", 
    "vitamin-b2", 
    "vitamin-b6", 
    "vitamin-b9", 
    "vitamin-b12", 
    "vitamin-c", 
    "vitamin-d", 
    "vitamin-e", 
    "vitamin-k", 
    "vitamin-pp", 
    "vitamine-h"
	]})

nutriments_categorized["nutriments"].append({"Énergie": [
    "energy", 
	"energy-kj", 
	"energy-kcal"
	]})

nutriments_categorized["nutriments"].append({"Valeurs nutritionnelles": [
    "fat", 
    "saturated-fat", 
    "carbohydrates", 
    "sugars", 
    "fiber", 
    "proteins", 
    "salt", 
    "sodium", 
    "cholesterol"
	]})

nutriments_categorized["nutriments"].append({"Minéraux": [
    "calcium", 
    "iron", 
    "magnesium", 
    "potassium", 
    "zinc"
	]})

nutriments_categorized["nutriments"].append({"Oméga": [
    "omega-3-fat", 
    "omega-6-fat", 
    "omega-9-fat"
	]})

nutriments_categorized["nutriments"].append({"Autres": [
    "caffeine", 
    "alcohol", 
    "water"
	]})

print(nutriments_categorized)

In [ ]:
unit_set = set()

def extraire_depuis_liste(liste_nutriments, nom_nutriment):
    if not isinstance(liste_nutriments, (np.ndarray, list)):
        return None

    for nutriment in liste_nutriments:
        if isinstance(nutriment, dict) and nutriment.get('name') == nom_nutriment:

            value_100g = nutriment.get('100g')
            if not isinstance(value_100g, str) and not isinstance(value_100g, float):
                return None

            unit = nutriment.get('unit')
            if unit == "&#181;g":
                unit = 'µg'
            elif unit == "% vol / *":
                unit = '% vol'
            elif unit == "kJ":
                unit = 'kj'
            elif unit == "":
                unit = None

            unit_set.add(unit)
            value_100g = float(value_100g)
            if unit == "g":
                value_100g = value_100g * 1.0
            elif unit == "mg":
                value_100g = value_100g * 0.001
            elif unit == "µg":
                value_100g = value_100g * 0.000001

            return value_100g
        
    return None

# Application sur le DataFrame Pandas
off_light_extended_df_fr = off_light_df_fr.copy()
for name in my_set:
    off_light_extended_df_fr[name] = off_light_extended_df_fr['nutriments'].apply(lambda x: extraire_depuis_liste(x, name))



In [ ]:
for u in unit_set:
    print(u)

In [ ]:
off_light_extended_df_fr.drop(columns=['nutriments'], inplace=True)
print(off_light_extended_df_fr.head(50))

In [ ]:

# for name in my_set:
#     avant = off_light_extended_df_fr[name].isna().sum()
#     off_light_extended_df_fr[name] = pd.to_numeric(off_light_extended_df_fr[name], errors="coerce")
#     apres = off_light_extended_df_fr[name].isna().sum()
#     print(f"{name} -> {avant}/{apres} ({apres - avant})")

    # pd.to_numeric(off_light_extended_df_fr[name], errors="coerce")
    # col = off_light_extended_df_fr[name]
    # foireuses = col.notna() & col.ne("") & pd.to_numeric(col, errors="coerce").isna()
    # off_light_extended_df_fr.loc[foireuses, name]

# counts = off_light_df_fr.groupby("code")["code"].count()
# print(off_light_extended_df_fr.head(10))

In [ ]:
off_light_extended_df_fr.to_parquet(food_light_extended_parquet, index=False)
# print(off_light_extended_df_fr.head(10))



In [ ]:
completion_per_group = {}
for main_cat in nutriments_categorized:
    for sub_cat in nutriments_categorized[main_cat]:
        for s in sub_cat:
            completion_per_group[s] = []

def find_category(name):
    for main_cat in nutriments_categorized:
        for sub_cat in nutriments_categorized[main_cat]:
            for s in sub_cat:
                for cat in sub_cat[s]:
                    if name == cat:
                        return s
    return ""

total_product = len(off_light_extended_df_fr)
print(f"Nombre de produit dans la liste: {total_product}")

output_sub_cat = []
for name in my_set:
    count = (~off_light_extended_df_fr[name].isna()).sum()
    # nan_count = off_light_extended_df_fr[name].isna().sum()
    # output_sub_cat.append((name, count, nan_count))
    output_sub_cat.append((name, count))

print()
output_sub_cat = sorted(output_sub_cat, key=lambda count: count[1], reverse=True)
for name, count in output_sub_cat:
    group = find_category(name)
    percent = count / total_product * 100.0

    print(f"{name} ({group}) => {count} => {percent}")

    completion_per_group[group].append(percent)

output_main_cat = []
for key in completion_per_group:
    output_main_cat.append((key, np.mean(completion_per_group[key])))

print()
output_main_cat = sorted(output_main_cat, key=lambda count: count[1], reverse=True)
for name, ratio in output_main_cat:
    print(f"{name} => {ratio}")


In [4]:
distri_and_cardinality_col = ['brands',
    'code',
    'completeness',
    # 'countries_tags',
    # 'entry_dates_tags',
    # 'food_groups_tags',
    # 'images',
    # 'ingredients_analysis_tags',
    # 'ingredients_original_tags',
    # 'ingredients_tags',
    # 'ingredients_text',
    'ingredients',
    'lang',
    # 'nova_groups_tags',
    # 'nutriments',
    'nutriscore_grade',
    'nutriscore_score',
    'obsolete',
    # 'origins_tags',
    # 'packaging_recycling_tags',
    # 'packaging_shapes_tags',
    # 'popularity_tags',
    # 'product_name',
    'product_quantity',
    'quantity'
    # 'vitamins_tags',
    # 'categories_properties',
	# 'allergens_tags'
    ]

In [13]:
distri_and_cardinality_dict = {}
distri_and_cardinality_output = []


In [14]:
import time

# On calcule le masque UNE SEULE FOIS
filters_df = pd.read_parquet(food_parquet, columns=["countries_tags", "nutriscore_grade"])

# mask = (filters_df["countries_tags"].apply(lambda x: filter_by_country(x, "en:france"))
#     & filters_df["nutriscore_grade"].apply(lambda x: filter_by_nutriscore(x)))

mask = filters_df["countries_tags"].apply(lambda x: filter_by_country(x, "en:france"))

print(f"Produits retenus : {mask.sum()} / {len(mask)}")

# Profiling
for d_c in distri_and_cardinality_col:
    if d_c not in distri_and_cardinality_dict:
        start = time.time()

        # On ne lit plus que la colonne à analyser
        tmp = pd.read_parquet(food_parquet, columns=[d_c])
        # On applique le masque déjà calculé
        tmp = tmp.loc[mask]

        print(f"{d_c} -> {tmp[d_c].dtype} " f"({time.time() - start:.2f} s)")

        # Cardinalité + distribution
        counts = tmp[d_c].value_counts()

        distri_and_cardinality_output.append((d_c, len(counts), (counts / counts.sum() * 100).head(5)))

        distri_and_cardinality_dict[d_c] = 1

Produits retenus : 1247336 / 4636471
brands -> str (1.40 s)
code -> str (1.35 s)
completeness -> float32 (1.23 s)
ingredients -> str (5.96 s)
lang -> str (1.77 s)
nutriscore_grade -> str (1.32 s)
nutriscore_score -> float64 (1.27 s)
obsolete -> bool (1.27 s)
product_quantity -> str (1.54 s)
quantity -> str (1.54 s)


In [15]:
# print(distri_and_cardinality_output)

total_product = mask.sum()
print(total_product)
for brand, count, values in distri_and_cardinality_output:
    print(f"{brand} -> {count} / {count / total_product * 100.0}")

1247336
brands -> 113784 / 9.122161149842544
code -> 1247309 / 99.9978353867763
completeness -> 64 / 0.005130935048775952
ingredients -> 319998 / 25.65451490215948
lang -> 81 / 0.0064938396711070635
nutriscore_grade -> 7 / 0.0005611960209598697
nutriscore_score -> 70 / 0.005611960209598697
obsolete -> 1 / 8.017086013712425e-05
product_quantity -> 4429 / 0.3550767395473233
quantity -> 28870 / 2.3145327321587765
